# Rolling-window analysis

A single full-history curve can hide start-date dependence. This notebook defaults to real Shiller data. The warm-up treatment in this exploratory notebook remains under mathematical review; do not use it as the final ranking source.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import plotly.express as px

from retail_sp500.backtest import BacktestConfig, run_backtest
from retail_sp500.data import load_shiller_data, synthetic_market_data
from retail_sp500.strategies import select_strategies

pd.set_option("display.max_columns", 100)

In [ ]:
USE_SYNTHETIC = False
market = synthetic_market_data(periods=1_200) if USE_SYNTHETIC else load_shiller_data()

print({
    "source": "synthetic smoke test" if USE_SYNTHETIC else "real Shiller monthly data",
    "start": market.index.min(),
    "end": market.index.max(),
    "rows": len(market),
})
assert USE_SYNTHETIC or market.index.max() <= pd.Timestamp.today().normalize()

strategy_keys = ["buy-hold", "trend-10m", "vol-target-12", "trend-vol-12"]
strategies, skipped = select_strategies(strategy_keys, market, skip_unavailable=True)

config = BacktestConfig(
    initial_cash=100_000,
    monthly_contribution=1_000,
    cash_annual_return=0.0,
)

skipped

## Rolling helper

This remains an exploratory orchestration layer. It will be replaced when common-start and pre-roll signal semantics are repaired in the package.

In [ ]:
def rolling_results(
    market: pd.DataFrame,
    strategies: list,
    config: BacktestConfig,
    *,
    window_years: int = 20,
    step_months: int = 12,
) -> pd.DataFrame:
    window_months = window_years * 12
    if window_months > len(market):
        raise ValueError("rolling window is longer than the dataset")
    if step_months < 1:
        raise ValueError("step_months must be positive")

    records = []
    for start in range(0, len(market) - window_months + 1, step_months):
        window = market.iloc[start : start + window_months]
        for strategy in strategies:
            result = run_backtest(window, strategy, config)
            records.append({
                "strategy": result.name,
                "start": window.index[0],
                "end": window.index[-1],
                **result.metrics,
            })
    return pd.DataFrame.from_records(records)

rolling = rolling_results(
    market,
    strategies,
    config,
    window_years=20,
    step_months=12,
)

rolling.head()

## Distribution summary

In [ ]:
summary = rolling.groupby("strategy").agg(
    windows=("ending_value", "size"),
    median_ending_value=("ending_value", "median"),
    tenth_percentile_ending_value=("ending_value", lambda values: values.quantile(0.10)),
    median_cagr=("time_weighted_cagr", "median"),
    median_max_drawdown=("max_drawdown", "median"),
    worst_max_drawdown=("max_drawdown", "min"),
)

summary.sort_values("median_ending_value", ascending=False)

In [ ]:
px.box(
    rolling,
    x="strategy",
    y="ending_value",
    points=False,
    title="Rolling 20-year terminal wealth",
).show()

In [ ]:
px.line(
    rolling,
    x="start",
    y="ending_value",
    color="strategy",
    title="Terminal wealth by rolling start date",
).show()